In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE paysim AS
SELECT *
FROM read_csv_auto('../data/paysim_transactions.csv');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
row_count = con.execute("""
SELECT COUNT(*) AS total_rows
FROM paysim;
""").df()

row_count

,total_rows
0,6362620


In [4]:
rule_preview = con.execute("""
SELECT
    type,
    amount,
    oldbalanceOrg,
    isFraud,

    CASE
        WHEN type IN ('TRANSFER', 'CASH_OUT')
             AND amount >= 200000
        THEN 1
        ELSE 0
    END AS rule_200k_flag,

    CASE
        WHEN type IN ('TRANSFER', 'CASH_OUT')
             AND amount >= 500000
        THEN 1
        ELSE 0
    END AS rule_500k_flag,

    CASE
        WHEN type IN ('TRANSFER', 'CASH_OUT')
             AND amount >= 500000
             AND oldbalanceOrg > 0
             AND amount >= oldbalanceOrg * 0.90
        THEN 1
        ELSE 0
    END AS rule_500k_depletion_flag

FROM paysim
LIMIT 20;
""").df()

rule_preview

,type,amount,oldbalanceOrg,isFraud,rule_200k_flag,rule_500k_flag,rule_500k_depletion_flag
0,PAYMENT,9839.64,170136.00,0,0,0,0
1,PAYMENT,1864.28,21249.00,0,0,0,0
2,TRANSFER,181.00,181.00,1,0,0,0
3,CASH_OUT,181.00,181.00,1,0,0,0
4,PAYMENT,11668.14,41554.00,0,0,0,0
5,PAYMENT,7817.71,53860.00,0,0,0,0
6,PAYMENT,7107.77,183195.00,0,0,0,0
7,PAYMENT,7861.64,176087.23,0,0,0,0
8,PAYMENT,4024.36,2671.00,0,0,0,0
9,DEBIT,5337.77,41720.00,0,0,0,0


In [5]:
rule_comparison = con.execute("""
WITH rule_flags AS (
    SELECT
        isFraud,

        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount >= 200000
            THEN 1
            ELSE 0
        END AS rule_200k_flag,

        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount >= 500000
            THEN 1
            ELSE 0
        END AS rule_500k_flag,

        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount >= 500000
                 AND oldbalanceOrg > 0
                 AND amount >= oldbalanceOrg * 0.90
            THEN 1
            ELSE 0
        END AS rule_500k_depletion_flag
    FROM paysim
),

rule_results AS (
    SELECT
        'Rule 1: Type + Amount >= 200K' AS rule_name,
        rule_200k_flag AS rule_flag,
        isFraud
    FROM rule_flags

    UNION ALL

    SELECT
        'Rule 2: Type + Amount >= 500K' AS rule_name,
        rule_500k_flag AS rule_flag,
        isFraud
    FROM rule_flags

    UNION ALL

    SELECT
        'Rule 3: Type + Amount >= 500K + 90% Depletion' AS rule_name,
        rule_500k_depletion_flag AS rule_flag,
        isFraud
    FROM rule_flags
)

SELECT
    rule_name,
    SUM(rule_flag) AS flagged_transactions,
    SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) AS true_positives,
    SUM(CASE WHEN rule_flag = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
    SUM(CASE WHEN rule_flag = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives,

    ROUND(
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) * 1.0
        / SUM(isFraud),
        4
    ) AS recall,

    ROUND(
        SUM(CASE WHEN rule_flag = 1 AND isFraud = 1 THEN 1 ELSE 0 END) * 1.0
        / SUM(rule_flag),
        4
    ) AS precision

FROM rule_results
GROUP BY rule_name
ORDER BY precision DESC;
""").df()

rule_comparison

,rule_name,flagged_transactions,true_positives,false_positives,false_negatives,recall,precision
0,Rule 3: Type + Amount >= 500K + 90% Depletion,158363.0,3725.0,154638.0,4488.0,0.4535,0.0235
1,Rule 2: Type + Amount >= 500K,314301.0,3864.0,310437.0,4349.0,0.4705,0.0123
2,Rule 1: Type + Amount >= 200K,1197669.0,5471.0,1192198.0,2742.0,0.6661,0.0046


In [6]:
rule_comparison.to_csv("../outputs/rule_comparison_results.csv", index=False)